# Silver → Gold — Microdados INEP

Este notebook consome a camada Silver e gera **3 tabelas Gold** em Parquet prontas para uso em painel (Power BI, Metabase, etc).

| Tabela | Descrição |
|--------|----------|
| `dim_escola.parquet` | Dimensão de escolas — dados cadastrais únicos por escola |
| `fct_matriculas_por_escola_ano.parquet` | Fato — matrículas por escola/ano, por nível, sexo e raça/cor |
| `fct_docentes_turmas_por_escola_ano.parquet` | Fato — docentes, turmas e infraestrutura por escola/ano |

> **Pré-requisito:** Execute primeiro o notebook `notebooks/silver/bronze_to_silver.ipynb`.


## Passo 1: Importação de Módulos

In [ ]:
import sys
import os

# Garante que o diretório raiz do projeto está no path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.jobs.gold.silver_to_gold import (
    run, _read_silver, _build_dim_escola,
    _build_fct_matriculas, _build_fct_docentes_turmas
)
from src.config import settings
from src.utils.logger import get_logger
import pandas as pd
from pathlib import Path

logger = get_logger('notebook_gold')
print('✓ Módulos carregados com sucesso')

## Passo 2: Verificar Silver disponível

In [ ]:
silver_path = Path(settings.SILVER_DATA_PATH) / 'microdados_escola'
parquet_files = sorted(silver_path.glob('ano=*/microdados_escola.parquet'))

if not parquet_files:
    print('❌ Nenhum arquivo Silver encontrado!')
    print('   Execute primeiro: notebooks/silver/bronze_to_silver.ipynb')
else:
    anos = [f.parent.name for f in parquet_files]
    print(f'✓ Silver disponível: {len(parquet_files)} anos encontrados')
    print(f'  Anos: {anos}')

## Passo 3: Executar Pipeline Silver → Gold

> ⚠️ **Este processo carrega todos os anos da Silver em memória**. Pode demorar alguns minutos dependendo da quantidade de dados.

In [ ]:
run()

## Passo 4: Verificação e Análise das Tabelas Gold

In [ ]:
gold_path = Path(settings.GOLD_DATA_PATH)

tabelas = {
    'dim_escola': gold_path / 'dim_escola.parquet',
    'fct_matriculas': gold_path / 'fct_matriculas_por_escola_ano.parquet',
    'fct_docentes_turmas': gold_path / 'fct_docentes_turmas_por_escola_ano.parquet',
}

print(f'{'Tabela':<35} {'Registros':>12} {'Colunas':>10} {'Tamanho (MB)':>14}')
print('-' * 75)
for nome, path in tabelas.items():
    if path.exists():
        df_v = pd.read_parquet(path)
        size_mb = path.stat().st_size / 1_048_576
        print(f'{nome:<35} {len(df_v):>12,} {len(df_v.columns):>10} {size_mb:>13.2f}')
    else:
        print(f'{nome:<35} ❌ NÃO ENCONTRADA')

In [ ]:
# -----------------------------------------------
# Inspeção: dim_escola
# -----------------------------------------------
df_dim = pd.read_parquet(gold_path / 'dim_escola.parquet')

print('=== dim_escola ===')
print(f'Escolas únicas: {df_dim["cod_escola"].nunique():,}')
print(f'\nDistribuição por Dependência:')
print(df_dim['desc_dependencia'].value_counts().to_string())
print(f'\nDistribuição por Localização:')
print(df_dim['desc_localizacao'].value_counts().to_string())
print(f'\nEscolas por Região:')
print(df_dim['nome_regiao'].value_counts().to_string())

In [ ]:
# -----------------------------------------------
# Inspeção: fct_matriculas_por_escola_ano
# -----------------------------------------------
df_mat = pd.read_parquet(gold_path / 'fct_matriculas_por_escola_ano.parquet')

print('=== fct_matriculas_por_escola_ano ===')
print(f'Registros: {len(df_mat):,} | Anos: {sorted(df_mat["ano_censo"].unique())}')
print(f'\nTotal de matrículas por ano:')
print(
    df_mat.groupby('ano_censo')['qt_matriculas_total']
    .sum()
    .apply(lambda x: f'{x:,.0f}')
    .to_string()
)
print(f'\nSchema:')
print(df_mat.dtypes.to_string())

In [ ]:
# -----------------------------------------------
# Inspeção: fct_docentes_turmas_por_escola_ano
# -----------------------------------------------
df_doc = pd.read_parquet(gold_path / 'fct_docentes_turmas_por_escola_ano.parquet')

print('=== fct_docentes_turmas_por_escola_ano ===')
print(f'Registros: {len(df_doc):,} | Anos: {sorted(df_doc["ano_censo"].unique())}')
print(f'\nTotal de docentes por ano:')
print(
    df_doc.groupby('ano_censo')['qt_docentes_total']
    .sum()
    .apply(lambda x: f'{x:,.0f}')
    .to_string()
)
print(f'\nSchema:')
print(df_doc.dtypes.to_string())

## Passo 5: Exemplos de queries para painel

Abaixo alguns exemplos de como consumir os dados Gold para criar visualizações.

In [ ]:
# Matrículas totais por UF no ano mais recente
df_dim = pd.read_parquet(gold_path / 'dim_escola.parquet')
df_mat = pd.read_parquet(gold_path / 'fct_matriculas_por_escola_ano.parquet')

ano_max = df_mat['ano_censo'].max()
df_mat_uf = (
    df_mat[df_mat['ano_censo'] == ano_max]
    .merge(df_dim[['cod_escola', 'uf_sigla', 'nome_uf', 'desc_dependencia']], on='cod_escola', how='left')
    .groupby(['uf_sigla', 'nome_uf'])['qt_matriculas_total']
    .sum()
    .reset_index()
    .sort_values('qt_matriculas_total', ascending=False)
)

print(f'Matrículas por UF — Censo {ano_max}:')
print(df_mat_uf.to_string(index=False))

In [ ]:
# Evolução de matrículas por nível de ensino (série histórica)
niveis = {
    'Infantil': 'qt_mat_infantil',
    'Fundamental': 'qt_mat_fundamental',
    'Médio': 'qt_mat_medio',
    'EJA': 'qt_mat_eja',
    'Especial': 'qt_mat_especial',
    'Prof. Técnico': 'qt_mat_prof_tecnico',
}

df_serie = df_mat.groupby('ano_censo')[list(niveis.values())].sum().reset_index()
df_serie = df_serie.rename(columns={v: k for k, v in niveis.items()})

print('Evolução de matrículas por nível (série histórica):')
print(df_serie.to_string(index=False))